# S06 — Exercises: SciPy, Pandas & Serialisation

**Python for Applied Physics** | Master in Applied Physics

| Symbol | Difficulty |
|--------|------------|
| 🟢 | Accessible |
| 🟡 | Intermediate |
| 🔴 | Challenging |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import h5py
import json
import tempfile, os
from scipy import integrate, optimize, interpolate, signal
from scipy import constants

%matplotlib inline
plt.rcParams['figure.dpi'] = 100
print("Setup complete")

---
## Exercise 1 🟢 — Beam power via `quad`

**Tasks:**
1. Use `scipy.integrate.quad` to compute the total power of a Gaussian beam with $w_0 = 350\,\mu\text{m}$ and peak intensity $I_0 = 1\,\text{MW/m}^2$.
2. Verify analytically: $P = I_0 \pi w_0^2 / 2$.
3. Compare `quad` precision to `np.trapz` on a radial grid of 50, 200, and 1000 points.
4. At what grid density does `trapz` match `quad` to better than $10^{-4}$ relative error?

In [ ]:
w0 = 350e-6   # m
I0 = 1e6      # W/m²

# 1. quad
def integrand(r):
    # YOUR CODE HERE — 2π r I(r)
    pass

P_quad, P_err = # YOUR CODE HERE

# 2. Analytic
P_analytic = # YOUR CODE HERE

# 3. trapz comparison
results = {}
for N_pts in [50, 200, 1000]:
    r_grid = np.linspace(0, 5*w0, N_pts)
    I_grid = I0 * np.exp(-2 * r_grid**2 / w0**2)
    P_trapz = # YOUR CODE HERE — 2π ∫ I(r) r dr
    rel_err  = abs(P_trapz - P_analytic) / P_analytic
    results[N_pts] = (P_trapz, rel_err)

# --- Assertions ---
assert P_quad is not None, "Implement integrand and quad call"
assert np.isclose(P_quad, P_analytic, rtol=1e-6), \
    f"quad result {P_quad:.6f} W ≠ analytic {P_analytic:.6f} W"

print(f"quad     : {P_quad:.8f} W  (err={P_err:.2e})")
print(f"analytic : {P_analytic:.8f} W")
print()
for N_pts, (P_t, rel_e) in results.items():
    flag = '✓' if rel_e < 1e-4 else ' '
    print(f"trapz N={N_pts:5d}: {P_t:.6f} W  rel_err={rel_e:.2e} {flag}")
print("All assertions passed ✓")

---
## Exercise 2 🟢 — Gaussian fit to a noisy beam profile

A CCD camera records a 1D beam profile. The data is provided below (simulated).

**Tasks:**
1. Fit the data with a Gaussian model $I(r) = I_{\text{peak}} \exp\!\left(-2(r-r_0)^2/w^2\right)$ using `curve_fit`.
2. Extract $I_{\text{peak}}$, $w$, and $r_0$ with their $1\sigma$ uncertainties.
3. Plot data, fit, and the $1\sigma$ confidence band (use `pcov`).
4. Verify that the true parameters (provided) lie within $2\sigma$ of the fitted values.

In [ ]:
# Simulate data
rng     = np.random.default_rng(123)
w_true  = 280e-6    # true beam radius
I_true  = 2.5e6     # true peak intensity (W/m²)
r0_true = 15e-6     # true centroid offset

r_data  = np.linspace(-1e-3, 1e-3, 100)
I_data  = I_true * np.exp(-2*(r_data - r0_true)**2 / w_true**2)
I_data += rng.normal(0, 0.04 * I_true, size=len(r_data))

# 1. Gaussian model
def gaussian_model(r, I_peak, w, r0):
    # YOUR CODE HERE
    pass

# 2. Fit
p0 = [I_data.max(), 300e-6, 0.0]
popt, pcov = # YOUR CODE HERE
perr       = # YOUR CODE HERE  — 1σ uncertainties

I_fit, w_fit, r0_fit   = popt
σ_I,   σ_w,   σ_r0     = perr

# 3. Plot
r_dense = np.linspace(r_data[0], r_data[-1], 512)
I_fitted = gaussian_model(r_dense, *popt)

# Confidence band: vary each parameter by ±1σ and take envelope
# (simple approach: propagate uncertainty analytically)
# YOUR PLOT CODE HERE
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(r_data*1e3, I_data/1e6, 'o', ms=3, alpha=0.6, label='Data')
ax.plot(r_dense*1e3, I_fitted/1e6, lw=2, label=f'Fit  w={w_fit*1e6:.0f}±{σ_w*1e6:.0f} µm')
ax.set_xlabel('r (mm)')
ax.set_ylabel('Intensity (MW/m²)')
ax.legend()
ax.set_title('Gaussian beam fit')
fig.tight_layout()
plt.show()

# 4. Assertions
assert gaussian_model is not None
assert popt is not None
assert np.abs(w_fit  - w_true)  < 2 * σ_w,  f"w  not within 2σ: {w_fit*1e6:.1f} vs {w_true*1e6:.1f} µm"
assert np.abs(r0_fit - r0_true) < 2 * σ_r0, f"r0 not within 2σ: {r0_fit*1e6:.1f} vs {r0_true*1e6:.1f} µm"
assert np.abs(I_fit  - I_true)  < 2 * σ_I,  f"I  not within 2σ"

print(f"I_peak = {I_fit/1e6:.3f} ± {σ_I/1e6:.3f} MW/m²  (true: {I_true/1e6:.3f})")
print(f"w      = {w_fit*1e6:.1f} ± {σ_w*1e6:.1f} µm      (true: {w_true*1e6:.0f})")
print(f"r0     = {r0_fit*1e6:.1f} ± {σ_r0*1e6:.1f} µm    (true: {r0_true*1e6:.0f})")
print("All assertions passed ✓")

---
## Exercise 3 🟡 — Upsample a coarse camera image

A camera has only $20 \times 20$ pixels. Use 2D interpolation to produce a smooth $200 \times 200$ image.

**Tasks:**
1. Build the coarse $20 \times 20$ Gaussian intensity map ($w_0 = 3$ pixels, $I_0 = 1$) — add Poisson noise: `rng.poisson(I_coarse * 1000) / 1000`.
2. Upsample to $200 \times 200$ with `RegularGridInterpolator` using `method='linear'` and `method='cubic'`.
3. Plot all three: coarse, linear upsampled, cubic upsampled.
4. Compute the RMS error of each against the true $200 \times 200$ Gaussian.
5. Which method is better, and why?

In [ ]:
rng = np.random.default_rng(55)

N_c  = 20    # coarse pixels
N_f  = 200   # fine pixels
w_px = 3     # beam radius in pixels

x_c = np.linspace(-10, 10, N_c)   # pixel coordinates
y_c = np.linspace(-10, 10, N_c)

# 1. Coarse image with Poisson noise
Xc, Yc = np.meshgrid(x_c, y_c, indexing='ij')
I_true_c = np.exp(-2*(Xc**2 + Yc**2) / w_px**2)
I_coarse = # YOUR CODE HERE — Poisson noise

# 2. Interpolators
x_f = np.linspace(-10, 10, N_f)
y_f = np.linspace(-10, 10, N_f)
Xf, Yf = np.meshgrid(x_f, y_f, indexing='ij')
pts_f   = np.column_stack([Xf.ravel(), Yf.ravel()])

I_true_f = np.exp(-2*(Xf**2 + Yf**2) / w_px**2)

results_interp = {}
for method in ['linear', 'cubic']:
    rgi = # YOUR CODE HERE
    I_up = rgi(pts_f).reshape(N_f, N_f)
    rms  = np.sqrt(np.mean((I_up - I_true_f)**2))
    results_interp[method] = (I_up, rms)

# 3. Plot
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
titles = ['Coarse (20×20)', 'Linear (200×200)', 'Cubic (200×200)']
images = [
    (I_coarse.T,                        x_c, y_c),
    (results_interp['linear'][0].T,  x_f, y_f),
    (results_interp['cubic'][0].T,   x_f, y_f),
]
for ax, title, (img, xax, yax) in zip(axes, titles, images):
    ax.imshow(img, origin='lower',
              extent=[xax[0], xax[-1], yax[0], yax[-1]],
              cmap='inferno', aspect='equal', vmin=0, vmax=1)
    ax.set_title(title)
    ax.set_xlabel('x (px)')
    ax.set_ylabel('y (px)')
fig.tight_layout()
plt.show()

# 4 & 5. Report
for method, (_, rms) in results_interp.items():
    print(f"{method:8s}: RMS error = {rms:.5f}")

# --- Assertions ---
assert I_coarse.shape == (N_c, N_c)
assert results_interp['linear'][0].shape  == (N_f, N_f)
assert results_interp['cubic'][0].shape   == (N_f, N_f)
assert results_interp['cubic'][1] <= results_interp['linear'][1], \
    "Cubic should be at least as accurate as linear for a smooth Gaussian"
print("All assertions passed ✓")

**Which method is better and why?**  
*Your answer here.*

---
## Exercise 4 🟡 — Multi-pulse detection + spectrogram

**Tasks:**
1. Build a time trace containing 4 Gaussian pulses at random positions within $[0, 30]\,\text{ps}$, with random amplitudes in $[0.5, 1.0]$ and $\tau = 300\,\text{fs}$. Add Gaussian noise (SNR ≈ 20 dB).
2. Use `scipy.signal.find_peaks` to detect all 4 pulses. Report their positions and amplitudes.
3. Apply a linear chirp ($\text{GDD} = 5000\,\text{fs}^2$) to the strongest pulse only.
4. Compute the spectrogram of the full trace with `scipy.signal.spectrogram` and display it with `pcolormesh`.
5. Verify that the chirped pulse appears as a diagonal ridge and the others appear as vertical streaks.

In [ ]:
rng    = np.random.default_rng(77)
N_tr   = 16384
dt_tr  = 2e-15
t_tr   = np.arange(N_tr) * dt_tr   # 0 to ~33 ps
τ_pulse = 300e-15
GDD_chirp = 5000e-30

# 1. Generate 4 pulses
t_positions = rng.uniform(5e-12, 25e-12, size=4)
amplitudes  = rng.uniform(0.5, 1.0, size=4)

I_trace = np.zeros(N_tr)
for t0, A in zip(t_positions, amplitudes):
    I_trace += # YOUR CODE HERE

# Noise at SNR=20 dB
noise_std = 10**(-20/20)
I_trace  += rng.normal(0, noise_std, size=N_tr)

# 3. Apply chirp to strongest pulse in frequency domain
idx_strongest = np.argmax(amplitudes)
t0_chirp      = t_positions[idx_strongest]
A_chirp       = amplitudes[idx_strongest]

# Extract the chirped pulse as a separate field, apply GDD, add back
E_single  = A_chirp * np.exp(-((t_tr - t0_chirp) / τ_pulse)**2)
freq_tr   = np.fft.fftfreq(N_tr, d=dt_tr)
ω_tr      = 2 * np.pi * freq_tr
E_f_single = np.fft.fft(E_single)
E_f_chirped = # YOUR CODE HERE — apply GDD phase
E_chirped   = # YOUR CODE HERE — IFFT

# Replace original pulse with chirped version
I_trace -= E_single**2
I_trace += np.abs(E_chirped)**2

# 2. Peak detection
min_dist = int(2e-12 / dt_tr)   # 2 ps minimum separation
peaks, props = # YOUR CODE HERE

# 4. Spectrogram
fs_tr = 1 / dt_tr
f_sp, t_sp, Sxx = # YOUR CODE HERE
t_sp_abs = t_sp   # already absolute since trace starts at 0

# --- Assertions ---
assert len(peaks) == 4, f"Expected 4 peaks, found {len(peaks)}"
assert Sxx.ndim == 2

print(f"Detected {len(peaks)} peaks:")
for i, pk in enumerate(peaks):
    print(f"  t = {t_tr[pk]*1e12:.2f} ps,  I = {I_trace[pk]:.3f}")

# 5. Plot
freq_mask = f_sp < 5e12
fig, axes = plt.subplots(2, 1, figsize=(10, 6))

axes[0].plot(t_tr*1e12, I_trace, lw=0.8, color='C0')
axes[0].plot(t_tr[peaks]*1e12, I_trace[peaks], 'v', ms=10, color='C1')
axes[0].set_xlabel('Time (ps)')
axes[0].set_ylabel('Intensity')
axes[0].set_title('Multi-pulse trace')

axes[1].pcolormesh(t_sp_abs*1e12, f_sp[freq_mask]/1e12,
                    Sxx[freq_mask, :], cmap='inferno', shading='auto')
axes[1].set_xlabel('Time (ps)')
axes[1].set_ylabel('Frequency (THz)')
axes[1].set_title('Spectrogram — chirped pulse visible as diagonal ridge')

fig.tight_layout()
plt.show()
print("All assertions passed ✓")

---
## Exercise 5 🟡 — Laser shot log analysis with Pandas

The CSV file below contains a simulated laser shot log from a multi-channel OPA.

**Tasks:**
1. Build the DataFrame from the provided dict (or load from CSV after writing it).
2. Report the number of shots per channel.
3. For each channel: compute mean energy, energy stability (std/mean × 100%), mean pulse duration.
4. Filter shots where energy deviates more than $2\sigma$ from the channel mean (outliers).
5. Compute the energy–duration correlation per channel using `.corr()`.

In [ ]:
rng = np.random.default_rng(42)
N   = 120

channels   = ['signal_1310nm', 'idler_1900nm', 'SH_655nm']
true_energy  = {'signal_1310nm': 30.0, 'idler_1900nm': 45.0, 'SH_655nm': 12.0}
true_dur     = {'signal_1310nm': 80.0, 'idler_1900nm': 120.0, 'SH_655nm': 60.0}

ch_col  = rng.choice(channels, size=N)
E_col   = np.array([rng.normal(true_energy[c], true_energy[c]*0.03) for c in ch_col])
dur_col = np.array([rng.normal(true_dur[c],   true_dur[c]*0.05)   for c in ch_col])

# Add a few outliers
outlier_idx = rng.choice(N, 5, replace=False)
E_col[outlier_idx] *= rng.uniform(0.5, 2.0, size=5)

df = pd.DataFrame({
    'shot_id' : np.arange(1, N+1),
    'channel' : ch_col,
    'energy_uJ': E_col.round(2),
    'duration_fs': dur_col.round(1),
})

# 2. Shots per channel
n_per_channel = # YOUR CODE HERE

# 3. Per-channel statistics
stats = # YOUR CODE HERE — groupby + agg: mean_energy, stability_pct, mean_duration

# 4. Flag outliers (|E - channel_mean| > 2σ)
df['channel_mean'] = df.groupby('channel')['energy_uJ'].transform('mean')
df['channel_std']  = df.groupby('channel')['energy_uJ'].transform('std')
df['is_outlier']   = # YOUR CODE HERE
n_outliers         = df['is_outlier'].sum()

# 5. Correlation per channel
correlations = df.groupby('channel')[['energy_uJ', 'duration_fs']].corr()

# --- Assertions ---
assert n_per_channel is not None
assert 'is_outlier' in df.columns
assert n_outliers >= 1, "Should have detected at least one outlier"

print("Shots per channel:")
print(n_per_channel)
print("\nPer-channel statistics:")
print(stats)
print(f"\nOutliers detected: {n_outliers}")
print(df[df['is_outlier']][['shot_id', 'channel', 'energy_uJ']].to_string())
print("All assertions passed ✓")

---
## Exercise 6 🟡 — Save/load a pulse dataset to HDF5

**Tasks:**
1. Build three Gaussian pulses with different durations ($\tau = 50, 100, 200\,\text{fs}$) and a common time axis.
2. Save them to a single HDF5 file with a hierarchical structure:
   ```
   pulse_dataset.h5
   ├── experiment/
   │   └── attrs: date, operator, laser_wavelength_nm
   ├── pulse_050fs/
   │   ├── t  (attrs: units='s')
   │   ├── E_t (attrs: units='a.u.')
   │   └── attrs: tau_fs=50
   ├── pulse_100fs/
   └── pulse_200fs/
   ```
3. Read back all three pulses and verify round-trip fidelity with `np.allclose`.
4. Plot the three intensity profiles from the loaded data (not the original arrays).
5. Add a `compression='gzip', compression_opts=6` argument to `create_dataset` and report the file size.

In [ ]:
N_h5  = 2048
dt_h5 = 5e-15
t_h5  = (np.arange(N_h5) - N_h5//2) * dt_h5

taus = {'pulse_050fs': 50e-15, 'pulse_100fs': 100e-15, 'pulse_200fs': 200e-15}
pulses = {name: np.exp(-t_h5**2 / (2*τ**2)) for name, τ in taus.items()}

with tempfile.TemporaryDirectory() as tmp:
    path_h5 = os.path.join(tmp, 'pulse_dataset.h5')

    # 2. Write
    with h5py.File(path_h5, 'w') as f:
        # Experiment group
        exp_grp = # YOUR CODE HERE
        exp_grp.attrs['date']                = '2026-04-01'
        exp_grp.attrs['operator']            = 'student'
        exp_grp.attrs['laser_wavelength_nm'] = 800.0

        # Pulse groups
        for name, τ_val in taus.items():
            grp = # YOUR CODE HERE — create group
            # YOUR CODE HERE — create datasets t and E_t with gzip compression
            grp.attrs['tau_fs'] = τ_val * 1e15

    file_size = os.path.getsize(path_h5)
    print(f"File size: {file_size/1024:.1f} kB")

    # 3. Read back and verify
    loaded = {}
    with h5py.File(path_h5, 'r') as f:
        f.visit(lambda name: print(f"  {name}"))
        for name in taus:
            grp = f[name]
            loaded[name] = {
                't'  : grp['t'][:],
                'E_t': grp['E_t'][:],
                'tau_fs': grp.attrs['tau_fs'],
            }

    for name, E_orig in pulses.items():
        ok = np.allclose(loaded[name]['E_t'], E_orig)
        print(f"{name}: round-trip {'✓' if ok else '✗'}")
        assert ok, f"{name} round-trip failed"

    # 4. Plot from loaded data
    fig, ax = plt.subplots(figsize=(7, 3.5))
    for name, data in loaded.items():
        ax.plot(data['t']*1e15, data['E_t']**2, lw=2,
                label=f"τ={data['tau_fs']:.0f} fs")
    ax.set_xlabel('Time (fs)')
    ax.set_ylabel('Intensity (a.u.)')
    ax.set_title('Pulses loaded from HDF5')
    ax.set_xlim(-600, 600)
    ax.legend()
    fig.tight_layout()
    plt.show()

print("All assertions passed ✓")

---
## Exercise 7 🔴 — Autocorrelation via FFT convolution

The intensity autocorrelation of a pulse is:

$$A(\tau) = \int_{-\infty}^{\infty} I(t)\,I(t - \tau)\,dt$$

It is measurable experimentally (e.g. with a Michelson interferometer + second-harmonic detector) and gives access to the pulse duration without direct time-resolution.

**Tasks:**
1. Compute $A(\tau)$ using the Wiener–Khinchin theorem: $A(\tau) = \mathcal{F}^{-1}\!\left[|\tilde{I}(\nu)|^2\right]$, where $\tilde{I}(\nu) = \mathcal{F}[I(t)]$.
2. Verify against `scipy.signal.correlate(I_t, I_t, mode='full')` — both should match.
3. For a Gaussian pulse $I(t) = \exp(-t^2/\tau^2)$, the autocorrelation is also Gaussian with width $\tau_{AC} = \sqrt{2}\,\tau$. Verify this numerically.
4. Repeat for a **chirped** pulse (GDD = $1000\,\text{fs}^2$). Does the autocorrelation width change? What does this tell you about the diagnostic?
5. Plot both autocorrelations (TL and chirped) on the same axes, normalised to their peak.

In [ ]:
N_ac  = 4096
dt_ac = 5e-15
τ_ac  = 100e-15
GDD_ac = 1000e-30

t_ac  = (np.arange(N_ac) - N_ac//2) * dt_ac
I_TL  = np.exp(-t_ac**2 / τ_ac**2)   # transform-limited intensity

# Build chirped pulse
freq_ac  = np.fft.fftshift(np.fft.fftfreq(N_ac, d=dt_ac))
ω_ac     = 2 * np.pi * freq_ac
E_ac     = np.exp(-t_ac**2 / (2*τ_ac**2))
E_f_ac   = np.fft.fftshift(np.fft.fft(E_ac))
E_f_ch   = E_f_ac * np.exp(0.5j * GDD_ac * ω_ac**2)
E_chirp_ac = np.fft.ifft(np.fft.ifftshift(E_f_ch))
I_chirp  = np.abs(E_chirp_ac)**2

def fwhm(x, y):
    yn = y / y.max(); above = yn >= 0.5
    return x[above].max() - x[above].min() if above.any() else np.nan

# 1. Autocorrelation via FFT (Wiener-Khinchin)
def autocorr_fft(I):
    """Compute intensity autocorrelation via the Wiener-Khinchin theorem."""
    # YOUR CODE HERE
    pass

A_TL    = # YOUR CODE HERE
A_chirp = # YOUR CODE HERE

# Build delay axis for FFT result (centred)
τ_axis  = np.fft.fftshift(np.fft.fftfreq(N_ac, d=1/(N_ac*dt_ac)))

# 2. Verify against scipy.signal.correlate
A_scipy = signal.correlate(I_TL, I_TL, mode='full')
τ_scipy = np.arange(-(N_ac-1), N_ac) * dt_ac

# 3. FWHM check
Δτ_TL_meas  = fwhm(τ_axis, np.fft.fftshift(A_TL))
Δτ_TL_expect = 2 * np.sqrt(2 * np.log(2)) * np.sqrt(2) * τ_ac   # τ_AC = sqrt(2)*τ FWHM

Δτ_chirp_meas = fwhm(τ_axis, np.fft.fftshift(A_chirp))

# --- Assertions ---
assert autocorr_fft is not None
assert A_TL is not None

# FFT vs scipy should match within numerical noise
A_TL_centred    = np.fft.fftshift(A_TL) / np.fft.fftshift(A_TL).max()
A_scipy_centred = A_scipy / A_scipy.max()
# Compare central region only
mid = N_ac // 2
assert np.allclose(A_TL_centred[mid-100:mid+100],
                   A_scipy_centred[N_ac-100:N_ac+100], atol=1e-4), \
    "FFT autocorr doesn't match scipy.signal.correlate"

assert np.isclose(Δτ_TL_meas, Δτ_TL_expect, rtol=0.02), \
    f"AC width {Δτ_TL_meas*1e15:.1f} fs ≠ expected {Δτ_TL_expect*1e15:.1f} fs"

# 5. Plot
A_TL_plt    = np.fft.fftshift(A_TL)
A_chirp_plt = np.fft.fftshift(A_chirp)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(τ_axis*1e15, A_TL_plt/A_TL_plt.max(),    lw=2, label=f'TL   FWHM={Δτ_TL_meas*1e15:.0f} fs')
ax.plot(τ_axis*1e15, A_chirp_plt/A_chirp_plt.max(), lw=2, ls='--',
        label=f'Chirped FWHM={Δτ_chirp_meas*1e15:.0f} fs')
ax.set_xlabel('Delay τ (fs)')
ax.set_ylabel('Autocorrelation (norm.)')
ax.set_title(f'Intensity autocorrelation  (GDD={GDD_ac*1e30:.0f} fs²)')
ax.set_xlim(-1500, 1500)
ax.legend()
fig.tight_layout()
plt.show()

print(f"TL   AC FWHM : {Δτ_TL_meas*1e15:.1f} fs  (expected {Δτ_TL_expect*1e15:.1f} fs)")
print(f"Chirped AC FWHM: {Δτ_chirp_meas*1e15:.1f} fs")
print()
print("What does this tell you about the autocorrelation as a diagnostic?")
print("→ YOUR ANSWER HERE")
print("\nAll assertions passed ✓")

---
## Wrap-up checklist

- [ ] All 7 exercises: assertions pass
- [ ] `shared/io.py` created with `save_pulse_hdf5` and `load_pulse_hdf5`
- [ ] Ex 7: written answer on what the autocorrelation tells you about chirped pulses
- [ ] `git add shared/io.py && git commit -m "S06: add HDF5 save/load to shared/io.py"`

**Next session:** S07 — Object-Oriented Programming